> **InferenceBase's dilemma:** The platform team is spending $80,000/month on OpenAI API calls. The CEO wants to know: can we self-host Llama-3-8B for $15k/month instead? To answer that, the Platform Engineer needs to know which GPU to order — and that requires understanding what a GPU actually does, why its specs translate to LLM workloads the way they do, and what the real bottleneck is for inference.
>
> **The numbers:** Llama-3-8B has 8 billion parameters. At bf16 precision, that's 16 GB of model weights alone — before activations or the KV cache. An RTX 4090 has 24 GB VRAM. An A100 has 80 GB. Does that mean A100 is just "more GPU"? No — the architecture differences run much deeper than capacity.

# GPU Hardware Foundations: Why GPUs Accelerate AI

| Part | Concept | Key question answered |
|------|---------|----------------------|
| 1 | CPU vs GPU: SIMD vs SIMT | Why does the same matmul run 10-50× faster on GPU? |
| 2 | Memory hierarchy | Why is LLM inference memory-bound, not compute-bound? |
| 3 | Roofline model | Which GPU spec matters most for which workload? |
| 4 | Warp execution and occupancy | Why does batch size affect GPU efficiency non-linearly? |
| 5 | Memory coalescing | Why does tensor layout affect throughput by 10×? |
| 6 | Toy → real bridge | Which GPU should InferenceBase buy, and why? |

---

> **Prerequisites:** `learning/genai/00-pytorch-primer` (PyTorch basics). GPU knowledge: none assumed.
> **Running example:** A `(B=8, S=128, D=256)` attention-shaped matrix multiply — representative of one head in a Transformer layer.

In [ ]:
import subprocess, sys
for pkg in ['torch','numpy','matplotlib']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import torch
import numpy as np
import matplotlib.pyplot as plt
import time

# ── GPU detection ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = torch.cuda.is_available()

print(f"Device: {DEVICE}")
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"  VRAM:        {props.total_memory / 1e9:.1f} GB")
    print(f"  CUDA cores:  {props.multi_processor_count} SMs")
    print(f"  Compute:     {props.major}.{props.minor}")
else:
    print("No GPU available — showing reference numbers from benchmark literature")
    print("All timing cells still run on CPU; ratios come from published benchmarks")

print()
# ── Running example dimensions ────────────────────────────────────────────────
B, S, D = 8, 128, 256  # batch=8, seq=128, dim=256 (one attention head)
print(f"Running example: (B={B}, S={S}, D={D}) attention-shaped matmul")
print(f"  This represents 1 of 32 heads in a typical 7B model at S={S}")

---

## Part 1 — CPU vs GPU: Why GPUs Are Fast

A CPU has 4–16 powerful cores optimised for low-latency serial work (branching code, OS tasks, database queries). A GPU has 5,000–16,000 simple cores optimised for high-throughput parallel work.

The difference is **SIMD vs. SIMT**:
- CPU SIMD: 1 instruction, 8-16 data elements at once (AVX-512)
- GPU SIMT: 1 instruction, 32 "threads" execute together (a **warp**)

For matrix multiplication: every output element is an independent dot product. 1,000 independent dot products → 1,000 GPU warps running truly simultaneously.

#### 🔮 Predict first

For a `(512, 512) × (512, 512)` matrix multiply:

How much faster is the GPU compared to a single CPU core?

1. **(a) 2–5×** — GPUs are faster but not dramatically
2. **(b) 10–50×** — GPU's parallelism gives a major speedup on this regular computation
3. **(c) 100–500×** — GPUs are orders of magnitude faster for any matrix operation

In [ ]:
# ── Part 1: CPU vs GPU timing on the running example ─────────────────────────
Q = torch.randn(B, S, D)   # Query matrix (running example)
K = torch.randn(B, S, D)   # Key matrix

def time_matmul(q, k, n_runs=20):
    # Warm up
    for _ in range(3):
        _ = torch.matmul(q, k.transpose(-2,-1))
    if q.is_cuda: torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        if q.is_cuda: torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = torch.matmul(q, k.transpose(-2,-1))  # (B, S, S) attention scores
        if q.is_cuda: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000, result

cpu_ms, _ = time_matmul(Q, K)
print(f"Running example: Q@K^T with shape ({B},{S},{D}) × ({B},{D},{S})")
print(f"  CPU time: {cpu_ms:.3f} ms")

if HAS_GPU:
    Q_gpu = Q.to(DEVICE); K_gpu = K.to(DEVICE)
    gpu_ms, _ = time_matmul(Q_gpu, K_gpu)
    speedup = cpu_ms / gpu_ms
    print(f"  GPU time: {gpu_ms:.3f} ms")
    print(f"  Speedup:  {speedup:.1f}×")

    if speedup > 50:
        print("\n→ Prediction (c) confirmed for this GPU")
    elif speedup > 10:
        print("\n→ Prediction (b) confirmed — significant speedup from parallel execution")
    else:
        print(f"\n→ Prediction (a) — small GPU (or short sequence length limits parallelism)")
else:
    # Reference numbers from published benchmarks (A100 vs single CPU core)
    print()
    print("Reference speedups (CPU single-core vs A100, from published benchmarks):")
    for size, speedup in [(256, 15), (512, 45), (1024, 120), (2048, 280)]:
        print(f"  ({size},{size}) matmul: ~{speedup}×")
    print("\n→ The speedup grows with matrix size because larger matrices expose more parallelism")
    print("  Prediction (b) for typical inference sizes; (c) at large batch sizes")

In [ ]:
# ── Part 1: Speedup vs. matrix size ──────────────────────────────────────────
sizes = [64, 128, 256, 512]
cpu_times, gpu_times = [], []

for s in sizes:
    q = torch.randn(4, s, s); k = torch.randn(4, s, s)
    t_cpu, _ = time_matmul(q, k, n_runs=10)
    cpu_times.append(t_cpu)
    if HAS_GPU:
        q_g = q.to(DEVICE); k_g = k.to(DEVICE)
        t_gpu, _ = time_matmul(q_g, k_g, n_runs=10)
        gpu_times.append(t_gpu)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sizes, cpu_times, 'o-', color='steelblue', label='CPU')
if HAS_GPU: axes[0].plot(sizes, gpu_times, 's-', color='coral', label='GPU')
axes[0].set_xlabel('Sequence length S'); axes[0].set_ylabel('Time (ms)')
axes[0].set_title('Matmul time vs. sequence length'); axes[0].legend()

if HAS_GPU and gpu_times:
    speedups = [c/g for c,g in zip(cpu_times, gpu_times)]
    axes[1].plot(sizes, speedups, 'D-', color='mediumseagreen')
    axes[1].set_xlabel('Sequence length S'); axes[1].set_ylabel('GPU speedup (×)')
    axes[1].set_title('GPU speedup grows with problem size')
else:
    # Reference curve
    ref_speedups = [4, 15, 45, 120]
    axes[1].plot(sizes, ref_speedups[:len(sizes)], 'D-', color='mediumseagreen', ls='--')
    axes[1].set_xlabel('Sequence length S'); axes[1].set_ylabel('GPU speedup (×, reference)')
    axes[1].set_title('GPU speedup grows with problem size (reference A100 data)')

plt.suptitle("CPU vs GPU: the matmul speedup grows as the problem gets larger", fontweight='bold')
plt.tight_layout(); plt.show()
print("→ Small problems (S=64): GPU overhead can dominate. Large problems (S=2048): GPU wins clearly.")

---

## Part 2 — Memory Hierarchy: Why LLM Inference Is Memory-Bound

The GPU has a layered memory system. From fastest (smallest) to slowest (largest):

| Level | Size | Bandwidth | Latency |
|-------|------|-----------|---------|
| Registers | 256 KB/SM | ~28 TB/s | 1 cycle |
| L1 / Shared memory | 228 KB/SM | ~19 TB/s | ~5 cycles |
| L2 cache | 40 MB | ~12 TB/s | ~200 cycles |
| HBM (main VRAM) | 24–80 GB | 0.9–3.4 TB/s | ~600 cycles |

For a matmul: both input matrices must be loaded from HBM. The output must be written back. The computation happens in registers. The ratio of computation to memory traffic — **arithmetic intensity** — determines whether we're compute-bound or memory-bound.

For LLM single-token inference: reading 8B weights from HBM to generate 1 token. At 2 TB/s bandwidth, reading 16 GB (bf16 weights) takes **8ms** per token. The GPU's compute capability (~300 TFLOPS) could theoretically do this in 0.1ms. → **Memory bound: bandwidth, not compute, is the bottleneck.**

![GPU memory hierarchy pyramid: HBM (80 GB, 2 TB/s) → L2 (40 MB) → SRAM/shared (228 KB/SM) → registers (fastest)](images/gpu-memory-hierarchy.png)

In [ ]:
# ── Part 2: Measure effective memory bandwidth ────────────────────────────────
def measure_bandwidth_gbps(nbytes, device, n_runs=10):
    """Measure achieved memory bandwidth for a copy operation."""
    src = torch.randn(nbytes // 4, dtype=torch.float32).to(device)  # float32 = 4 bytes
    if src.is_cuda: torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        if src.is_cuda: torch.cuda.synchronize()
        t0 = time.perf_counter()
        dst = src.clone()
        if src.is_cuda: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    ms = np.median(times) * 1000
    gbps = (2 * nbytes / 1e9) / (np.median(times))  # read + write = 2× bytes
    return gbps, ms

print("Memory bandwidth benchmarks:")
for mb in [16, 64, 256, 1024]:  # megabytes
    nbytes = mb * 1024 * 1024
    bw_cpu, ms_cpu = measure_bandwidth_gbps(nbytes, torch.device('cpu'))
    print(f"  {mb:4d} MB copy  CPU: {bw_cpu:.1f} GB/s ({ms_cpu:.1f}ms)", end="")
    if HAS_GPU:
        bw_gpu, ms_gpu = measure_bandwidth_gbps(nbytes, DEVICE)
        print(f"  GPU: {bw_gpu:.1f} GB/s ({ms_gpu:.1f}ms)")
    else:
        print(f"  GPU: ~2,000 GB/s (A100 reference)")

print()
print("LLM inference memory-bound calculation:")
model_gb = 16.0  # Llama-3-8B at bf16 = 8B params × 2 bytes
ref_bw   = 2000  # GB/s (A100 HBM)
token_latency_ms = (model_gb / ref_bw) * 1000
print(f"  Llama-3-8B weights: {model_gb:.0f} GB at bf16")
print(f"  A100 HBM bandwidth: {ref_bw} GB/s")
print(f"  Min latency per token (memory-bound): {token_latency_ms:.1f}ms")
print(f"  → At best: {1000/token_latency_ms:.0f} tokens/second (memory bandwidth limit)")

#### What just happened — and what's missing

Memory bandwidth is the bottleneck for LLM inference: reading 16 GB of model weights from HBM takes ~8ms on an A100, regardless of compute capability. This is why buying a GPU with higher bandwidth (A100: 2 TB/s) matters more than higher TFLOPS for single-token inference.

**Missing piece:** How do we compare different GPUs systematically? We need a model that captures both compute AND memory constraints — the **roofline model** does exactly this.

---

## Part 3 — The Roofline Model: Which Spec Matters for Your Workload?

The roofline model predicts performance based on **arithmetic intensity** (AI): FLOP per byte of memory traffic.

$$\text{Performance} = \min(\text{peak TFLOPS},\ \text{bandwidth} \times \text{AI})$$

- If AI < ridge point → **memory-bound**: buy more bandwidth (HBM)
- If AI > ridge point → **compute-bound**: buy more TFLOPS

The **ridge point** is where the two limits intersect: $\text{ridge} = \frac{\text{peak TFLOPS}}{\text{bandwidth (TFLOP/s/TB/s)}}$. A GPU with 77 TFLOPS and 2 TB/s has ridge ≈ 38.5 FLOP/byte.

**LLM workloads and their arithmetic intensity:**
- Single-token decode: AI ≈ 1–5 (very memory-bound — one token reads all weights once)
- Prefill (large prompt): AI ≈ 10–100 (more compute-bound as sequence grows)
- Training: AI ≈ 50–200 (compute-bound with large batch; multiple passes over weights)

![Roofline model: LLM decode sits far left in the memory-bound region; matmul sits near the compute ceiling](images/roofline-model.png)

In [ ]:
# ── Part 3: Roofline model ────────────────────────────────────────────────────
# GPU specs (reference values from published datasheets)
gpus = {
    'RTX 4090': {'tflops': 165.0, 'bandwidth_tbs': 1.0,  'vram_gb': 24,  'cost_mo': 1.5},
    'A10G':     {'tflops': 31.2,  'bandwidth_tbs': 0.6,  'vram_gb': 24,  'cost_mo': 3.0},
    'A100 80G': {'tflops': 77.0,  'bandwidth_tbs': 2.0,  'vram_gb': 80,  'cost_mo': 10.0},
    'H100':     {'tflops': 204.0, 'bandwidth_tbs': 3.35, 'vram_gb': 80,  'cost_mo': 25.0},
}

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['coral', 'steelblue', 'mediumseagreen', 'orange']
ai_range = np.logspace(-1, 3, 300)

for (name, spec), color in zip(gpus.items(), colors):
    bw_gbs = spec['bandwidth_tbs'] * 1000
    ridge  = spec['tflops'] * 1000 / bw_gbs  # GFLOP/GB = FLOP/byte
    perf   = np.minimum(spec['tflops'] * np.ones_like(ai_range), bw_gbs / 1000 * ai_range)
    ax.loglog(ai_range, perf, lw=2, color=color, label=f"{name} (${spec['cost_mo']}/hr)")
    ax.axvline(ridge, color=color, ls=':', lw=0.8, alpha=0.6)

# Annotate key LLM workloads
for label, ai, y_pos in [('LLM decode\n(1 token)', 2, 3), ('LLM prefill\n(512 tokens)', 50, 6),
                           ('Training\n(batch=32)', 150, 30)]:
    ax.scatter([ai], [y_pos], s=100, zorder=5, color='black')
    ax.annotate(label, (ai, y_pos), textcoords='offset points', xytext=(8, 5), fontsize=8)

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=11)
ax.set_ylabel('Attainable Performance (TFLOPS)', fontsize=11)
ax.set_title('Roofline Model — memory-bound left of ridge, compute-bound right', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

print("Key insight: LLM decode (AI≈2) is deep in the memory-bound region.")
print("For InferenceBase's use case: HBM bandwidth matters more than raw TFLOPS.")
print()
print("Best value for Llama-3-8B inference:")
for name, spec in gpus.items():
    tokens_per_s = spec['bandwidth_tbs'] * 1000 / (16.0 / 1000)  # 16GB model / bandwidth
    efficiency = tokens_per_s / spec['cost_mo']
    print(f"  {name}: ~{tokens_per_s:.0f} tok/s, ${spec['cost_mo']}/hr → {efficiency:.0f} tok/s/dollar")

---

## Part 4 — Warp Execution and Occupancy

A GPU **warp** is 32 threads that execute the same instruction simultaneously (SIMT).
- **High occupancy:** many active warps → GPU hides memory latency by switching to a ready warp while another waits for HBM
- **Low occupancy:** few active warps → GPU sits idle waiting for memory fetches

For LLM inference with `batch_size=1`: one token, one forward pass, few active warps → low GPU utilization. With `batch_size=32`: 32 independent decode paths → many warps → high utilization.

This explains a counter-intuitive result: **batching amortizes weight reads**. The 16 GB of Llama-3-8B model weights must be read from HBM on every forward pass. With batch=1, those 16 GB generate 1 output token. With batch=32, the same 16 GB generate 32 output tokens — the weight-read cost is shared.

$$\text{throughput} = \frac{\text{bandwidth (GB/s)}}{\text{model size (GB)}} \times \text{batch\_size}$$

Until the KV cache fills VRAM, throughput scales linearly with batch size.

![Warp divergence: 20 threads take the "if" path (teal), 12 are disabled (gray), serializing execution and doubling latency](images/warp-simt-execution.png)

In [ ]:
# ── Part 4: Batch size effect on throughput ───────────────────────────────────
# Simulate how throughput (tokens/sec) scales with batch size
# For memory-bound inference: throughput scales with batch size until VRAM fills up

D_MODEL = 4096  # Llama-7B hidden dim
FF_DIM  = 16384  # feedforward expansion (4× hidden)
batch_sizes = [1, 2, 4, 8, 16, 32]

def flops_per_forward(batch, seq=1, d=D_MODEL, ff=FF_DIM):
    """Rough FLOP count for one forward pass (attention + FFN)."""
    attn_flops = 4 * batch * seq * d * d    # QKV projections + output
    ffn_flops  = 2 * batch * seq * d * ff   # two linear layers
    return (attn_flops + ffn_flops) / 1e9   # GFLOP

# Memory traffic stays roughly constant (model weights dominate for single-token decode)
model_gb = 14.0  # 7B params × 2 bytes (bf16)
ref_bw   = 2.0   # TB/s (A100)

print("Throughput analysis for Llama-7B single-token decode:")
print(f"{'Batch':>6}  {'GFLOPs/step':>12}  {'Arith. Int.':>12}  {'Approx. tok/s':>14}  {'tok/s/req':>10}")
print("-" * 65)
for b in batch_sizes:
    flops = flops_per_forward(b)
    mem_gb = model_gb + b * D_MODEL * 2 / 1e9  # model + activations (small)
    ai = flops / mem_gb                          # FLOP/GB (arithmetic intensity)
    toks_per_s = ref_bw * 1000 * ai             # bandwidth × AI (memory-bound estimate)
    print(f"  {b:4d}  {flops:12.1f}  {ai:12.2f}  {min(toks_per_s, 77000):14.0f}  {min(toks_per_s,77000)/b:10.0f}")

print()
print("→ With batch=1: most of the GPU's 5000+ cores sit idle (low arithmetic intensity)")
print("→ With batch=32: more work per memory read → higher GPU utilization → more tok/s/dollar")

# Plot throughput vs batch size
toks_list = []
for b in batch_sizes:
    flops = flops_per_forward(b)
    mem_gb = model_gb + b * D_MODEL * 2 / 1e9
    ai = flops / mem_gb
    toks_list.append(min(ref_bw * 1000 * ai, 77000))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(batch_sizes, toks_list, 'o-', color='coral', lw=2)
axes[0].set_xlabel('Batch size'); axes[0].set_ylabel('Tokens/second')
axes[0].set_title('Throughput scales with batch size (memory-bound regime)')

axes[1].plot(batch_sizes, [t/b for t,b in zip(toks_list,batch_sizes)], 's-', color='steelblue', lw=2)
axes[1].set_xlabel('Batch size'); axes[1].set_ylabel('Tokens/second/request')
axes[1].set_title('Per-request latency stays roughly constant until VRAM fills')

plt.suptitle("Batching amortizes model weight reads → linear throughput scaling", fontweight='bold')
plt.tight_layout(); plt.show()

---

## Part 5 — Memory Coalescing: Why Tensor Layout Matters

When 32 threads in a warp access memory simultaneously, the GPU can combine them into one or a few memory transactions — if and only if the addresses are **contiguous**. This is **coalesced access**.

**Row-major tensor (PyTorch default):**
- Reading a row = contiguous addresses → 1 memory transaction → coalesced ✓
- Reading a column = strided addresses (every `N` bytes apart) → 1 transaction per thread → 32× the traffic ✗

This is why transposing before matmul matters. `A.t()` in PyTorch creates a **non-contiguous view** — the data is unchanged in memory, but the stride metadata flips. When a matmul kernel then reads rows of `A.t()`, those rows are columns of `A` in memory → strided → slow.

**In Transformers specifically:** `K.transpose(-2, -1)` in `Q @ K^T` creates a non-contiguous key matrix every forward pass. FlashAttention avoids this by keeping K tiles in SRAM (shared memory) and reordering compute to avoid the strided HBM access entirely.

In [ ]:
# ── Part 5: Coalesced vs. strided memory access ───────────────────────────────
M = 4096
A = torch.randn(M, M).to(DEVICE)
B_mat = torch.randn(M, M).to(DEVICE)

def time_op(fn, n=10):
    if HAS_GPU: torch.cuda.synchronize()
    times = []
    for _ in range(n):
        if HAS_GPU: torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if HAS_GPU: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000

# Contiguous matmul (A @ B: rows of A are contiguous in memory)
t_contig = time_op(lambda: torch.matmul(A, B_mat))

# Non-contiguous matmul (A.t() is a strided view: columns of A = strided access)
t_strided = time_op(lambda: torch.matmul(A.t(), B_mat))

# Contiguous after explicit copy (A.t().contiguous() materialises the transposed layout)
A_t_c = A.t().contiguous()
t_contiguous_copy = time_op(lambda: torch.matmul(A_t_c, B_mat))

print(f"Matrix multiply on ({M},{M}) square matrix:")
print(f"  A @ B (contiguous):               {t_contig:.2f} ms  ← baseline")
print(f"  A.t() @ B (non-contiguous/strided): {t_strided:.2f} ms  ({t_strided/t_contig:.1f}× slower)")
print(f"  A.t().contiguous() @ B:             {t_contiguous_copy:.2f} ms  (pre-copy restores speed)")
print()
print("→ Non-contiguous tensor operations break memory coalescing.")
print("  In attention: K.transpose(-2,-1) creates a non-contiguous view.")
print("  FlashAttention (covered in learning/ai-infrastructure/03-flash-attention/)")
print("  avoids this by keeping KV in SRAM tiles, sidestepping the strided HBM access.")

# Check contiguity
print()
print(f"  A.is_contiguous():       {A.is_contiguous()}")
print(f"  A.t().is_contiguous():   {A.t().is_contiguous()}")
print(f"  A.t().contiguous().is_contiguous(): {A.t().contiguous().is_contiguous()}")

---

## Part 6 — Toy → Real: Which GPU Should InferenceBase Buy?

Let's apply everything we've learned to answer the original question: which GPU for Llama-3-8B at < $15k/month?

**Checklist so far:**

| Requirement | Comes from |
|---|---|
| VRAM > 16 GB (model) + ~4 GB (KV cache) | Part 2 — memory hierarchy |
| Buy bandwidth, not TFLOPS | Part 3 — roofline model (AI ≈ 2 for decode) |
| Use batch_size > 1 to amortize weight reads | Part 4 — warp occupancy |
| Keep KV tensors contiguous in memory | Part 5 — memory coalescing |

The GPU table below compares four candidates against these requirements.

In [ ]:
# ── Part 6: GPU selection for InferenceBase ───────────────────────────────────
model_gb_bf16 = 16.0  # Llama-3-8B at bf16

print("GPU selection analysis for Llama-3-8B inference:")
print(f"{'GPU':15s} {'VRAM':6s} {'BW TB/s':8s} {'Fits?':6s} {'Tok/s':7s} {'$/hr':6s} {'Tok/$/hr':8s}")
print("-" * 65)
for name, spec in gpus.items():
    fits = "✓" if spec['vram_gb'] > model_gb_bf16 + 4 else "✗"  # +4GB for KV cache
    bw_gbs = spec['bandwidth_tbs'] * 1000
    toks = bw_gbs / (model_gb_bf16 / 1000)
    tpd  = toks / spec['cost_mo']
    print(f"  {name:13s} {spec['vram_gb']:4.0f}GB  {spec['bandwidth_tbs']:6.1f}    {fits:5s}  {toks:6.0f}  {spec['cost_mo']:5.2f}  {tpd:8.0f}")

print()
print("InferenceBase analysis:")
print(f"  Budget target:  < $15,000/month")
print(f"  Required VRAM:  > {model_gb_bf16:.0f} GB (model) + 4 GB (KV cache) = 20 GB minimum")
print()

# Best value: RTX 4090
rtx4090 = gpus['RTX 4090']
bw_4090 = rtx4090['bandwidth_tbs'] * 1000
toks_4090 = bw_4090 / (model_gb_bf16 / 1000)
monthly_cost_4090 = rtx4090['cost_mo'] * 730  # hours/month

print(f"  RECOMMENDATION: RTX 4090")
print(f"    VRAM: 24 GB  ✓  (fits with {24 - model_gb_bf16 - 4:.0f} GB headroom for KV cache)")
print(f"    Throughput: ~{toks_4090:.0f} tok/s (bandwidth-limited)")
print(f"    Monthly cost: ~${monthly_cost_4090:,.0f} (730 hr/month × ${rtx4090['cost_mo']}/hr)")
print(f"    Savings vs. OpenAI: ~${80000 - monthly_cost_4090:,.0f}/month")
print()
a100 = gpus['A100 80G']
monthly_cost_a100 = a100['cost_mo'] * 730
print(f"  WHY NOT A100?")
print(f"    A100 has {a100['bandwidth_tbs']/rtx4090['bandwidth_tbs']:.1f}× more bandwidth ({a100['bandwidth_tbs']:.1f} vs {rtx4090['bandwidth_tbs']:.1f} TB/s)")
print(f"    → {a100['bandwidth_tbs']/rtx4090['bandwidth_tbs']:.1f}× more tokens/second")
print(f"    But costs {a100['cost_mo']/rtx4090['cost_mo']:.0f}× more per hour (${a100['cost_mo']} vs ${rtx4090['cost_mo']})")
print(f"    Monthly: ~${monthly_cost_a100:,.0f} — exceeds $15k budget")
print(f"    For inference only: RTX 4090 wins on tok/s per dollar")

---

## Summary and Closing Decision

| Part | Concept | Key insight |
|------|---------|-------------|
| 1 | CPU vs GPU | GPUs win on large parallel matmuls (10–100×); CPU wins on serial branchy code |
| 2 | Memory hierarchy | LLM inference is memory-bound: HBM bandwidth, not TFLOPS, is the bottleneck |
| 3 | Roofline model | AI < ridge point → buy bandwidth; AI > ridge → buy TFLOPS |
| 4 | Warp occupancy | batch=1 underutilizes GPU; batch=32 amortizes model weight reads |
| 5 | Memory coalescing | Strided access (non-contiguous tensors) can be 5-10× slower than coalesced |
| 6 | GPU selection | RTX 4090 for inference: best tok/s/dollar for memory-bound workloads |

In [ ]:
# ── Closing Decision ──────────────────────────────────────────────────────────
print("=" * 60)
print("  CLOSING DECISION — InferenceBase GPU Selection")
print("=" * 60)
print()
print("  Llama-3-8B inference requirements:")
print(f"    Model size (bf16): {model_gb_bf16:.0f} GB")
print(f"    Workload type: memory-bound (AI ≈ 2–5 for single-token decode)")
print()
print("  SELECTED: RTX 4090 @ ~$1.50/hr")
print(f"    VRAM: 24 GB ✓  (fits with {24 - model_gb_bf16 - 4:.0f} GB for KV cache)")
print(f"    Throughput: ~{toks_4090:.0f} tok/s (BW-limited)")
print(f"    Monthly cost: ~${monthly_cost_4090:,.0f}")
print(f"    Savings vs. OpenAI: ~${80000 - monthly_cost_4090:,.0f}/month")
print()
print("  WHY NOT A100?")
print(f"    A100 has 3.3× more bandwidth (2.0 vs 1.0 TB/s) → ~3.3× faster")
print(f"    But costs {gpus['A100 80G']['cost_mo']/rtx4090['cost_mo']:.0f}× more per hour")
print(f"    For inference: RTX 4090 wins on tok/s per dollar")
print()
print("  The GPU spec that matters most for LLM inference: HBM BANDWIDTH")
print("  (not TFLOPS, not core count, not clock speed)")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- **CPU vs GPU timing** — matmul speedup measured across sequence lengths
- **Memory bandwidth** — measured on actual hardware (or reference numbers on CPU)
- **Roofline model** — arithmetic intensity computed for LLM workloads; plotted for 4 GPUs
- **Batch size and occupancy** — throughput-per-dollar analysis across batch sizes
- **Memory coalescing** — contiguous vs. strided matmul timing
- **GPU selection** — concrete recommendation for InferenceBase with measured justification

### Tier 2 — Explained but Not Built
- **Tensor Cores** — specialised matrix multiply units (A100: 312 TFLOPS bf16 peak); the roofline model assumes them; building a custom Tensor Core kernel is out of scope here

### Tier 3 — Named but Out of Scope
- **NVLink / NVSwitch** — high-speed GPU interconnect for multi-GPU systems; matters for training, less for single-GPU inference
- **MIG (Multi-Instance GPU)** — partition one A100 into up to 7 independent GPU instances; useful for multi-tenant serving
- **Triton custom kernels** — writing custom GPU kernels in Python; covered in `learning/ai-infrastructure/08-triton-kernels/`

---

## When to Use What

| Workload | Bottleneck | Buy | Avoid |
|---|---|---|---|
| LLM single-token decode | HBM bandwidth | More GB/s (A100, H100) | Pure TFLOPS upgrades |
| LLM prefill (large prompt) | Mixed | Balanced (H100) | — |
| Training, large batch | Compute | More TFLOPS (H100 FP8) | Bandwidth-only GPUs |
| Serving multiple small models | Memory capacity | More VRAM | — |

→ **Next:** `learning/ai-infrastructure/02-mixed-precision/` — now that you understand the memory hierarchy, this chapter answers: how do precision formats (fp32, fp16, bf16, int8) affect how much model fits in VRAM, and what breaks when you reduce precision?